# ChatGPT NLP + LSTM 多因子投资策略

## 论文参考

- **标题**: Multi-factor Investment Strategy Based on ChatGPT and Deep Learning
- **作者**: Ruien Zhang
- **年份**: 2023

### 摘要

本文提出融合 ChatGPT 生成的 NLP 情绪因子与传统技术因子，输入 LSTM 深度学习模型进行股价预测。
本 notebook 模拟 NLP 情绪因子 (使用与收益率相关的随机过程作为代理)，
将其与动量、波动率等传统因子合并后送入 LSTM 模型。

> **⚠️ 教学演示声明**
> 
> 本 Notebook 为量化策略算法教学样例，回测结果包含**前视偏差 (Look-ahead Bias)**：
> - 训练标签使用了未来收益率（`shift(-N)`）
> - StandardScaler 等预处理可能在全量数据上拟合
> - 模型参数选择可能基于完整样本期
> 
> **所有回测收益率仅供学习参考，不代表策略的实际可期望表现，不可直接用于实盘交易。**

## 算法原理

### NLP 情绪因子 + LSTM 融合

**1. NLP 管线 (模拟)**

$$\text{新闻文本} \xrightarrow{\text{Tokenize}} \text{Token序列} \xrightarrow{\text{Embedding}} \text{向量} \xrightarrow{\text{Sentiment Model}} \text{情绪得分} \in [-1, 1]$$

由于不调用实际 NLP 模型，我们生成与收益率有噪声相关的模拟情绪因子。

**2. 多因子融合**

$$X_t = [\text{mom}_t, \text{vol}_t, \text{rsi}_t, \text{macd}_t, \text{sentiment}_t, \ldots]$$

**3. LSTM 预测模型**

$$h_t = \text{LSTM}(X_t, h_{t-1})$$
$$\hat{r}_{t+1} = W_o \cdot h_t + b_o$$

使用过去 30 天多因子序列预测下一日收益率方向。

**4. 交易信号**

- $\hat{r}_{t+1} > 0$: 做多
- $\hat{r}_{t+1} < 0$: 空仓

In [ ]:
# ============================================================
# Cell 3: 动画展示 - NLP + LSTM 管线流程
# ============================================================
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 创建分步展示的NLP管线动画
pipeline_stages = [
    ('News\nText', '#795548'),
    ('Tokenize\n& Embed', '#2196F3'),
    ('Sentiment\nScore', '#4CAF50'),
    ('Traditional\nFactors', '#FF9800'),
    ('Factor\nMerge', '#9C27B0'),
    ('LSTM\nModel', '#F44336'),
    ('Prediction\n& Signal', '#00BCD4'),
]

n = len(pipeline_stages)
frames = []

for step in range(1, n + 1):
    shapes = []
    annotations = []
    for i in range(step):
        name, color = pipeline_stages[i]
        shapes.append(dict(
            type='rect', x0=i-0.38, x1=i+0.38, y0=-0.4, y1=0.4,
            fillcolor=color, opacity=0.85, line=dict(color='white', width=2)
        ))
        annotations.append(dict(x=i, y=0, text=f'<b>{name}</b>', showarrow=False,
                                font=dict(size=10, color='white')))
        if i > 0:
            annotations.append(dict(
                x=i-0.42, y=0, ax=i-0.58, ay=0,
                arrowhead=2, arrowsize=1.5, arrowwidth=2, arrowcolor='gray',
                showarrow=True, text=''
            ))
    frames.append(go.Frame(
        data=[go.Scatter(x=[], y=[])],
        layout=go.Layout(shapes=shapes, annotations=annotations),
        name=f'Step {step}'
    ))

fig = go.Figure(
    data=[go.Scatter(x=[], y=[])],
    frames=frames,
    layout=go.Layout(
        title=dict(text='NLP情绪因子 + LSTM 预测管线'),
        xaxis=dict(visible=False, range=[-0.7, n-0.3]),
        yaxis=dict(visible=False, range=[-1, 1]),
        height=300, width=950, plot_bgcolor='white',
        shapes=frames[0].layout.shapes,
        annotations=frames[0].layout.annotations,
        updatemenus=[dict(type='buttons', buttons=[
            dict(label='\u25b6 播放', method='animate',
                 args=[None, {'frame': {'duration': 700}}]),
        ])],
    )
)
fig.show()

In [ ]:
# ============================================================
# Cell 4: 环境设置与导入
# ============================================================
import sys
import warnings
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
np.random.seed(42)

sys.path.insert(0, '..')
from shared.data_fetcher import get_etf_daily, get_index_daily
from shared.backtest_engine import Backtester
from shared.factors import momentum, volatility, rsi, macd, bollinger_bands
from shared.visualization import (
    plot_equity_curve, plot_drawdown, plot_metrics_table,
    plot_returns_distribution
)
from shared.constants import DEFAULT_START, DEFAULT_END, ETF_CSI300

# PyTorch
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'所有模块导入成功, device={device}')

In [ ]:
# ============================================================
# Cell 5: 数据获取 + 模拟NLP情绪因子
# ============================================================
try:
    df = get_etf_daily(ETF_CSI300, DEFAULT_START, DEFAULT_END)
    print(f'ETF 510300 数据: {len(df)} 条')
except Exception as e:
    print(f'数据获取异常: {e}, 使用模拟数据')
    dates = pd.bdate_range(DEFAULT_START, DEFAULT_END)
    n = len(dates)
    np.random.seed(42)
    price = 4.5 + np.cumsum(np.random.randn(n) * 0.02)
    price = np.maximum(price, 2.0)
    df = pd.DataFrame({
        'open': price * (1 + np.random.randn(n) * 0.003),
        'close': price,
        'high': price * (1 + np.abs(np.random.randn(n) * 0.008)),
        'low': price * (1 - np.abs(np.random.randn(n) * 0.008)),
        'volume': np.random.randint(50000, 2000000, n).astype(float),
    }, index=dates)

# 生成模拟NLP情绪因子
# 真实场景: ChatGPT分析新闻 -> 情绪得分
# 模拟: 与未来收益有噪声相关的随机过程
actual_returns = df['close'].pct_change().shift(-1)  # 未来收益
noise = np.random.randn(len(df)) * 0.5
raw_sentiment = 0.3 * actual_returns.fillna(0).values + 0.7 * noise  # 30% 信号 + 70% 噪声
# 平滑并归一化到 [-1, 1]
sentiment = pd.Series(raw_sentiment, index=df.index)
sentiment = sentiment.rolling(3).mean().fillna(0)
sentiment = sentiment / sentiment.abs().max()  # 归一化

df['sentiment'] = sentiment
print(f'情绪因子统计: mean={sentiment.mean():.4f}, std={sentiment.std():.4f}')
print(f'数据范围: {df.index[0].strftime("%Y-%m-%d")} ~ {df.index[-1].strftime("%Y-%m-%d")}')

In [ ]:
# ============================================================
# Cell 6: 因子工程 + 数据集构建
# ============================================================

# 构建多因子
features = pd.DataFrame(index=df.index)

# 传统因子
mom = momentum(df['close'], [5, 10, 20])
features = pd.concat([features, mom], axis=1)

vol = volatility(df['close'], [5, 10, 20])
features = pd.concat([features, vol], axis=1)

features['rsi_14'] = rsi(df['close'], 14)
macd_df = macd(df['close'])
features['macd_hist'] = macd_df['histogram']
bb = bollinger_bands(df['close'])
features['bb_pctb'] = bb['bb_pctb']

# 量价因子
features['volume_chg'] = df['volume'].pct_change(5)
features['price_range'] = (df['high'] - df['low']) / df['close']

# NLP 情绪因子
features['sentiment'] = df['sentiment']
features['sentiment_ma3'] = df['sentiment'].rolling(3).mean()
features['sentiment_ma5'] = df['sentiment'].rolling(5).mean()

# 标签: 下一日收益率 (正为1, 负为0)
labels = (df['close'].pct_change().shift(-1) > 0).astype(float)

# 清洗
features = features.dropna()
labels = labels.reindex(features.index).dropna()
common_idx = features.index.intersection(labels.index)
features = features.loc[common_idx]
labels = labels.loc[common_idx]

FEATURE_COLS = features.columns.tolist()
print(f'因子数量: {len(FEATURE_COLS)}')
print(f'因子列表: {FEATURE_COLS}')
print(f'样本数: {len(features)}')

# 标准化
scaler = StandardScaler()
X_scaled = scaler.fit_transform(features.values)

# 构建LSTM序列数据
SEQ_LEN = 30

def create_sequences(X, y, seq_len):
    Xs, ys, indices = [], [], []
    for i in range(seq_len, len(X)):
        Xs.append(X[i-seq_len:i])
        ys.append(y[i])
        indices.append(i)
    return np.array(Xs), np.array(ys), indices

X_seq, y_seq, seq_indices = create_sequences(X_scaled, labels.values, SEQ_LEN)
print(f'LSTM 输入形状: {X_seq.shape}  (samples, seq_len, features)')

# 划分训练/测试 (前70%训练, 后30%测试)
split = int(len(X_seq) * 0.7)
X_train, X_test = X_seq[:split], X_seq[split:]
y_train, y_test = y_seq[:split], y_seq[split:]
test_indices = seq_indices[split:]

print(f'训练集: {X_train.shape[0]}, 测试集: {X_test.shape[0]}')

In [ ]:
# ============================================================
# Cell 7: LSTM 模型训练
# ============================================================

class LSTMClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_layers=2, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers,
                            batch_first=True, dropout=dropout)
        self.fc1 = nn.Linear(hidden_dim, 32)
        self.fc2 = nn.Linear(32, 1)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]  # 取最后时步
        out = self.dropout(self.relu(self.fc1(out)))
        return self.sigmoid(self.fc2(out)).squeeze(-1)


# 转为PyTorch张量
X_tr_t = torch.FloatTensor(X_train).to(device)
y_tr_t = torch.FloatTensor(y_train).to(device)
X_te_t = torch.FloatTensor(X_test).to(device)

train_ds = TensorDataset(X_tr_t, y_tr_t)
train_dl = DataLoader(train_ds, batch_size=64, shuffle=True)

# 模型
model = LSTMClassifier(input_dim=len(FEATURE_COLS), hidden_dim=64, num_layers=2).to(device)
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)

# 训练
EPOCHS = 50
train_losses = []

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0
    n_batch = 0
    for xb, yb in train_dl:
        pred = model(xb)
        loss = criterion(pred, yb)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        epoch_loss += loss.item()
        n_batch += 1
    scheduler.step()
    avg_loss = epoch_loss / n_batch
    train_losses.append(avg_loss)
    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1}/{EPOCHS}, Loss: {avg_loss:.4f}')

# 预测
model.eval()
with torch.no_grad():
    pred_probs = model(X_te_t).cpu().numpy()

pred_labels = (pred_probs > 0.5).astype(int)
accuracy = (pred_labels == y_test).mean()
print(f'\n测试集准确率: {accuracy:.2%}')

In [ ]:
# ============================================================
# Cell 8: 信号生成与回测
# ============================================================

# 将预测转为交易信号
test_dates = features.index[test_indices]
signals = pd.Series(0, index=test_dates)
signals[pred_labels == 1] = 1   # 预测上涨 -> 做多
signals[pred_labels == 0] = 0   # 预测下跌 -> 空仓

# 获取基准
try:
    bench = get_index_daily('sh000300', DEFAULT_START, DEFAULT_END)
    bench_prices = bench['close']
except:
    bench_prices = None

backtester = Backtester(initial_capital=1_000_000, t_plus_1=True)
result = backtester.run(df['close'], signals, bench_prices)

print('=== ChatGPT NLP + LSTM 策略回测结果 ===')
for k, v in result['metrics'].items():
    print(f'  {k}: {v}')

In [ ]:
# ============================================================
# Cell 9: 绩效可视化
# ============================================================
import matplotlib.pyplot as plt

# 1. 收益曲线
benchmark_equity = None
if result.get('benchmark_returns') is not None:
    benchmark_equity = (1 + result['benchmark_returns']).cumprod() * result['equity_curve'].iloc[0]
plot_equity_curve(result['equity_curve'], benchmark_equity,
                  title='ChatGPT NLP + LSTM - 累计收益')

# 2. 回撤
plot_drawdown(result['equity_curve'], title='NLP+LSTM - 回撤')

# 3. 训练损失 + 情绪因子分布
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(train_losses, color='#F44336', linewidth=2)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('LSTM 训练损失')
ax1.grid(True, alpha=0.3)

ax2.hist(df['sentiment'].dropna(), bins=50, color='#4CAF50', alpha=0.7, density=True)
ax2.set_xlabel('情绪得分')
ax2.set_ylabel('密度')
ax2.set_title('模拟 NLP 情绪因子分布')
ax2.axvline(x=0, color='red', linestyle='--', alpha=0.5)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 4. 绩效表
plot_metrics_table(result['metrics'], title='NLP+LSTM 绩效指标')

## 结果讨论

### 策略表现

1. **NLP 因子贡献**: 模拟情绪因子为模型提供了额外的预测维度
2. **LSTM 优势**: 能捕捉因子序列中的时序依赖关系
3. **多因子融合**: 传统技术因子 + NLP 因子的组合效果通常优于单一类型

### 模拟限制

- NLP 情绪因子是人工构造的代理变量，与真实 ChatGPT 输出有差距
- 真实情绪分析需考虑: 新闻时效性、信息重叠、情绪过度反应
- LSTM 模型可能过拟合较短的训练序列

### 改进方向

- 接入真实 NLP 模型处理财经新闻/社交媒体数据
- 加入 Attention 机制提升 LSTM 对关键时步的关注
- 使用 Transformer 替代 LSTM 处理长序列
- 多资产截面预测替代单资产时序预测